In [1]:
import pandas as pd
import numpy as np

import torch
from torch import nn

from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
pd.set_option('display.max_rows', 1000)

## 读取数据

In [3]:
train_data = pd.read_csv('/Users/macbook/Projects/d2l_project/pytorch/3.chapter_multilayer-perceptrons/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/Users/macbook/Projects/d2l_project/pytorch/3.chapter_multilayer-perceptrons/house-prices-advanced-regression-techniques/test.csv')

In [4]:
train_data.head()
train_data.shape
test_data.head()
test_data.shape

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


(1460, 81)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


(1459, 80)

## 预处理

In [5]:
labels = train_data['SalePrice']
features = train_data.drop(columns=['SalePrice'])
labels.head()
features.head()

0    208500
1    181500
2    223500
3    140000
4    250000
Name: SalePrice, dtype: int64

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


In [6]:
all_feature = pd.concat([features, test_data])
features.shape
test_data.shape
all_feature.shape

(1460, 80)

(1459, 80)

(2919, 80)

In [7]:
features.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal


In [8]:
numeric_feature = all_feature.dtypes[
    all_feature.dtypes != 'object'
].index

In [9]:
numeric_feature

Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
       'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
       'MiscVal', 'MoSold', 'YrSold'],
      dtype='object')

In [10]:
all_feature[numeric_feature] = (all_feature[numeric_feature] - all_feature[numeric_feature].mean()) / all_feature[numeric_feature].std()

In [11]:
all_feature.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,-1.731161,0.067320,RL,-0.184443,-0.217841,Pave,NaN,Reg,Lvl,AllPub,...,-0.285886,-0.063139,NaN,NaN,NaN,-0.089577,-1.551918,0.157619,WD,Normal
1,-1.729974,-0.873466,RL,0.458096,-0.072032,Pave,NaN,Reg,Lvl,AllPub,...,-0.285886,-0.063139,NaN,NaN,NaN,-0.089577,-0.446848,-0.602858,WD,Normal
2,-1.728788,0.067320,RL,-0.055935,0.137173,Pave,NaN,IR1,Lvl,AllPub,...,-0.285886,-0.063139,NaN,NaN,NaN,-0.089577,1.026577,0.157619,WD,Normal
3,-1.727601,0.302516,RL,-0.398622,-0.078371,Pave,NaN,IR1,Lvl,AllPub,...,-0.285886,-0.063139,NaN,NaN,NaN,-0.089577,-1.551918,-1.363335,WD,Abnorml
4,-1.726415,0.067320,RL,0.629439,0.518814,Pave,NaN,IR1,Lvl,AllPub,...,-0.285886,-0.063139,NaN,NaN,NaN,-0.089577,2.131647,0.157619,WD,Normal


In [12]:
all_feature = all_feature.fillna(0)
all_feature

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,-1.731161,0.067320,RL,-0.184443,-0.217841,Pave,0,Reg,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,-1.551918,0.157619,WD,Normal
1,-1.729974,-0.873466,RL,0.458096,-0.072032,Pave,0,Reg,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,-0.446848,-0.602858,WD,Normal
2,-1.728788,0.067320,RL,-0.055935,0.137173,Pave,0,IR1,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,1.026577,0.157619,WD,Normal
3,-1.727601,0.302516,RL,-0.398622,-0.078371,Pave,0,IR1,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,-1.551918,-1.363335,WD,Abnorml
4,-1.726415,0.067320,RL,0.629439,0.518814,Pave,0,IR1,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,2.131647,0.157619,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,1.726415,2.419286,RM,-2.069222,-1.043758,Pave,0,Reg,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,-0.078492,-1.363335,WD,Normal
1455,1.727601,2.419286,RM,-2.069222,-1.049083,Pave,0,Reg,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,-0.815205,-1.363335,WD,Abnorml
1456,1.728788,-0.873466,RL,3.884968,1.246594,Pave,0,Reg,Lvl,AllPub,...,-0.285886,-0.063139,0,0,0,-0.089577,1.026577,-1.363335,WD,Abnorml
1457,1.729974,0.655311,RL,-0.312950,0.034599,Pave,0,Reg,Lvl,AllPub,...,-0.285886,-0.063139,0,MnPrv,Shed,1.144116,0.289865,-1.363335,WD,Normal


In [13]:
all_feature = pd.get_dummies(
    all_feature,
    dtype=int
)
all_feature

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,-1.731161,0.067320,-0.184443,-0.217841,0.646073,-0.507197,1.046078,0.896679,0.523038,0.580708,...,0,0,0,1,0,0,0,0,1,0
1,-1.729974,-0.873466,0.458096,-0.072032,-0.063174,2.187904,0.154737,-0.395536,-0.569893,1.177709,...,0,0,0,1,0,0,0,0,1,0
2,-1.728788,0.067320,-0.055935,0.137173,0.646073,-0.507197,0.980053,0.848819,0.333448,0.097840,...,0,0,0,1,0,0,0,0,1,0
3,-1.727601,0.302516,-0.398622,-0.078371,0.646073,-0.507197,-1.859033,-0.682695,-0.569893,-0.494771,...,0,0,0,1,1,0,0,0,0,0
4,-1.726415,0.067320,0.629439,0.518814,1.355319,-0.507197,0.947040,0.753100,1.381770,0.468770,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,1.726415,2.419286,-2.069222,-1.043758,-1.481667,1.289537,-0.043338,-0.682695,-0.569893,-0.968860,...,0,0,0,1,0,0,0,0,1,0
1455,1.727601,2.419286,-2.069222,-1.049083,-1.481667,-0.507197,-0.043338,-0.682695,-0.569893,-0.415757,...,0,0,0,1,1,0,0,0,0,0
1456,1.728788,-0.873466,3.884968,1.246594,-0.772420,1.289537,-0.373465,0.561660,-0.569893,1.717643,...,0,0,0,1,1,0,0,0,0,0
1457,1.729974,0.655311,-0.312950,0.034599,-0.772420,-0.507197,0.682939,0.370221,-0.569893,-0.229194,...,0,0,0,1,0,0,0,0,1,0


In [14]:
n_train = len(features)
train_feature = all_feature[0:n_train]
test_feature = all_feature[n_train:]

In [15]:
train_feature
test_feature
train_feature['Id'] = train_feature.index + 1
test_feature['Id'] = test_feature.index + 1
train_feature
test_feature

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,-1.731161,0.067320,-0.184443,-0.217841,0.646073,-0.507197,1.046078,0.896679,0.523038,0.580708,...,0,0,0,1,0,0,0,0,1,0
1,-1.729974,-0.873466,0.458096,-0.072032,-0.063174,2.187904,0.154737,-0.395536,-0.569893,1.177709,...,0,0,0,1,0,0,0,0,1,0
2,-1.728788,0.067320,-0.055935,0.137173,0.646073,-0.507197,0.980053,0.848819,0.333448,0.097840,...,0,0,0,1,0,0,0,0,1,0
3,-1.727601,0.302516,-0.398622,-0.078371,0.646073,-0.507197,-1.859033,-0.682695,-0.569893,-0.494771,...,0,0,0,1,1,0,0,0,0,0
4,-1.726415,0.067320,0.629439,0.518814,1.355319,-0.507197,0.947040,0.753100,1.381770,0.468770,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,-0.004746,0.067320,-0.312950,-0.285421,-0.063174,-0.507197,0.914028,0.753100,-0.569893,-0.968860,...,0,0,0,1,0,0,0,0,1,0
1456,-0.003560,-0.873466,0.672275,0.381246,-0.063174,0.391170,0.220763,0.178782,0.093673,0.765076,...,0,0,0,1,0,0,0,0,1,0
1457,-0.002373,0.302516,-0.141607,-0.142781,0.646073,3.086271,-1.000704,1.040259,-0.569893,-0.365275,...,0,0,0,1,0,0,0,0,1,0
1458,-0.001187,-0.873466,-0.055935,-0.057197,-0.772420,0.391170,-0.703591,0.561660,-0.569893,-0.861312,...,0,0,0,1,0,0,0,0,1,0


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,0.001187,-0.873466,0.458096,0.184340,-0.772420,0.391170,-0.340452,-1.113434,-0.569893,0.058332,...,0,0,0,1,0,0,0,0,1,0
1,0.002373,-0.873466,0.500932,0.519702,-0.063174,0.391170,-0.439490,-1.257014,0.032335,1.056991,...,0,0,0,1,0,0,0,0,1,0
2,0.003560,0.067320,0.201080,0.464294,-0.772420,-0.507197,0.848003,0.657380,-0.569893,0.767271,...,0,0,0,1,0,0,0,0,1,0
3,0.004746,0.067320,0.372424,-0.024105,-0.063174,0.391170,0.881015,0.657380,-0.458369,0.352443,...,0,0,0,1,0,0,0,0,1,0
4,0.005933,1.478499,-1.126832,-0.654636,1.355319,-0.507197,0.682939,0.370221,-0.569893,-0.391613,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,1.726415,2.419286,-2.069222,-1.043758,-1.481667,1.289537,-0.043338,-0.682695,-0.569893,-0.968860,...,0,0,0,1,0,0,0,0,1,0
1455,1.727601,2.419286,-2.069222,-1.049083,-1.481667,-0.507197,-0.043338,-0.682695,-0.569893,-0.415757,...,0,0,0,1,1,0,0,0,0,0
1456,1.728788,-0.873466,3.884968,1.246594,-0.772420,1.289537,-0.373465,0.561660,-0.569893,1.717643,...,0,0,0,1,1,0,0,0,0,0
1457,1.729974,0.655311,-0.312950,0.034599,-0.772420,-0.507197,0.682939,0.370221,-0.569893,-0.229194,...,0,0,0,1,0,0,0,0,1,0


/var/folders/8v/vs1dyzcd3tg7svjgmdvk2__h0000gn/T/ipykernel_96933/3728407080.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_feature['Id'] = train_feature.index + 1
/var/folders/8v/vs1dyzcd3tg7svjgmdvk2__h0000gn/T/ipykernel_96933/3728407080.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_feature['Id'] = test_feature.index + 1


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,0.067320,-0.184443,-0.217841,0.646073,-0.507197,1.046078,0.896679,0.523038,0.580708,...,0,0,0,1,0,0,0,0,1,0
1,2,-0.873466,0.458096,-0.072032,-0.063174,2.187904,0.154737,-0.395536,-0.569893,1.177709,...,0,0,0,1,0,0,0,0,1,0
2,3,0.067320,-0.055935,0.137173,0.646073,-0.507197,0.980053,0.848819,0.333448,0.097840,...,0,0,0,1,0,0,0,0,1,0
3,4,0.302516,-0.398622,-0.078371,0.646073,-0.507197,-1.859033,-0.682695,-0.569893,-0.494771,...,0,0,0,1,1,0,0,0,0,0
4,5,0.067320,0.629439,0.518814,1.355319,-0.507197,0.947040,0.753100,1.381770,0.468770,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,0.067320,-0.312950,-0.285421,-0.063174,-0.507197,0.914028,0.753100,-0.569893,-0.968860,...,0,0,0,1,0,0,0,0,1,0
1456,1457,-0.873466,0.672275,0.381246,-0.063174,0.391170,0.220763,0.178782,0.093673,0.765076,...,0,0,0,1,0,0,0,0,1,0
1457,1458,0.302516,-0.141607,-0.142781,0.646073,3.086271,-1.000704,1.040259,-0.569893,-0.365275,...,0,0,0,1,0,0,0,0,1,0
1458,1459,-0.873466,-0.055935,-0.057197,-0.772420,0.391170,-0.703591,0.561660,-0.569893,-0.861312,...,0,0,0,1,0,0,0,0,1,0


,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,-0.873466,0.458096,0.184340,-0.772420,0.391170,-0.340452,-1.113434,-0.569893,0.058332,...,0,0,0,1,0,0,0,0,1,0
1,2,-0.873466,0.500932,0.519702,-0.063174,0.391170,-0.439490,-1.257014,0.032335,1.056991,...,0,0,0,1,0,0,0,0,1,0
2,3,0.067320,0.201080,0.464294,-0.772420,-0.507197,0.848003,0.657380,-0.569893,0.767271,...,0,0,0,1,0,0,0,0,1,0
3,4,0.067320,0.372424,-0.024105,-0.063174,0.391170,0.881015,0.657380,-0.458369,0.352443,...,0,0,0,1,0,0,0,0,1,0
4,5,1.478499,-1.126832,-0.654636,1.355319,-0.507197,0.682939,0.370221,-0.569893,-0.391613,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,1455,2.419286,-2.069222,-1.043758,-1.481667,1.289537,-0.043338,-0.682695,-0.569893,-0.968860,...,0,0,0,1,0,0,0,0,1,0
1455,1456,2.419286,-2.069222,-1.049083,-1.481667,-0.507197,-0.043338,-0.682695,-0.569893,-0.415757,...,0,0,0,1,1,0,0,0,0,0
1456,1457,-0.873466,3.884968,1.246594,-0.772420,1.289537,-0.373465,0.561660,-0.569893,1.717643,...,0,0,0,1,1,0,0,0,0,0
1457,1458,0.655311,-0.312950,0.034599,-0.772420,-0.507197,0.682939,0.370221,-0.569893,-0.229194,...,0,0,0,1,0,0,0,0,1,0


## 转换成tensor

In [19]:
train_feature = torch.tensor(
    train_feature.values,
    dtype=torch.float32
)

TypeError: must be real number, not builtin_function_or_method

In [20]:
type(train_feature)

torch.Tensor

In [22]:
type(labels)
labels.shape

pandas.core.series.Series

(1460,)

In [23]:
train_labels = torch.tensor(
    labels.values,
    dtype=torch.float32
)
type(train_labels)

torch.Tensor

In [ ]:
train_labels.shape
train_labels.reshape(-1, 1)

torch.Size([1460])

## 创建dataloader

In [25]:
dataset = TensorDataset(
    train_feature,
    train_labels
)

In [26]:
dataset[1]

(tensor([ 2.0000, -0.8735,  0.4581, -0.0720, -0.0632,  2.1879,  0.1547, -0.3955,
         -0.5699,  1.1777, -0.2930, -0.6297,  0.4769,  0.2610, -0.7849, -0.1012,
         -0.4718, -0.8193,  3.8205,  0.7812, -0.7562,  0.1699, -0.2077, -0.2877,
          0.6235, -0.0826,  0.3064, -0.0598,  1.6146, -0.7027, -0.3595, -0.1033,
         -0.2859, -0.0631, -0.0896, -0.4468, -0.6029,  0.0000,  0.0000,  0.0000,
          0.0000,  1.0000,  0.0000,  0.0000,  1.0000,  1.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,  1.0000,
          0.0000,  1.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  0.0000,
          1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  

In [27]:
train_iter = DataLoader(
    dataset=dataset,
    batch_size=64,
    shuffle=True
)

## 定义模型

In [28]:
# 只是一个线性模型
def get_model(input_feartue):

    model = nn.Linear(
        input_feartue,
        1
    )
    return model

In [ ]:
# 损失函数
loss_fn = nn.MSELoss()

def log_rmse(y_hat, y):

    clipped_pred = torch.clamp(
        y_hat,
        1,
        float("inf")
    )

    rmse = torch.sqrt(
        loss_fn(
            torch.log(clipped_pred),
            torch.log(y)
        )
    )
    return rmse